<div class="alert alert-info">
    
➡️ Before you start, make sure that you are familiar with the **[study guide](https://liu-nlp.ai/text-mining/logistics/)**, in particular the rules around **cheating and plagiarism**.

➡️ Make sure you fill in **all** cells that say **`YOUR CODE HERE`** or **YOUR ANSWER HERE**.  You normally shouldn't need to modify any of the other cells.

➡️ Make sure that you save the notebook **with the outputs from all cells visible,** as if you had run the notebook from top to bottom.

</div>

# L3: Information Extraction

Information extraction (IE) is the task of identifying named entities and semantic relations between these entities in text data. In this lab, we will focus on two sub-tasks in information extraction:

1. **Named entity recognition (NER)**, i.e. identifying mentions of entities; and
2. **Entity linking**, i.e. matching these mentions to entities in a knowledge base.

In [1]:
from liu_tm import check_solutions

## Dataset

The main data set for this lab is a collection of news wire articles in which mentions of **named entities** have been annotated with **page names** from the [English Wikipedia](https://en.wikipedia.org/wiki/). The next code cell loads the training and the development parts of the data into Pandas data frames.

In [2]:
import bz2
import csv
import pandas as pd
import numpy as np

with bz2.open('ner-train.tsv.bz2', mode='rt', encoding='utf-8') as source:
    df_train = pd.read_csv(source, encoding='utf-8', sep='\t', quoting=csv.QUOTE_NONE)

with bz2.open('ner-dev.tsv.bz2', mode='rt', encoding='utf-8') as source:
    df_dev = pd.read_csv(source, encoding='utf-8', sep='\t', quoting=csv.QUOTE_NONE)

Each row in these two data frames corresponds to one mention of a named entity and has five columns:

1. a unique identifier for the sentence containing the entity mention
2. the pre-tokenized sentence, with tokens separated by spaces
3. the start position of the token span containing the entity mention
4. the end position of the token span (exclusive, as in Python list indexing)
5. the entity label; either a Wikipedia page name or the generic label `--NME--`

The following cell prints the first ten samples from the training data:

In [3]:
df_train.head(10)

,sentence_id,sentence,beg,end,label
0,0000-000,EU rejects German call to boycott British lamb .,0,1,--NME--
1,0000-000,EU rejects German call to boycott British lamb .,2,3,Germany
2,0000-000,EU rejects German call to boycott British lamb .,6,7,United_Kingdom
3,0000-001,Peter Blackburn,0,2,--NME--
4,0000-002,BRUSSELS 1996-08-22,0,1,Brussels
5,0000-003,The European Commission said on Thursday it di...,1,3,European_Commission
6,0000-003,The European Commission said on Thursday it di...,9,10,Germany
7,0000-003,The European Commission said on Thursday it di...,15,16,United_Kingdom
8,0000-004,Germany 's representative to the European Unio...,0,1,Germany
9,0000-004,Germany 's representative to the European Unio...,5,7,European_Union


For example, we can see that the first sentence is annotated with three entity mentions:

* the span 0–1 &lsquo;EU&rsquo; is annotated as an entity but only labelled with the generic `--NME--`
* the span 2–3 &lsquo;German&rsquo; is annotated with the page [Germany](http://en.wikipedia.org/wiki/Germany)
* the span 6–7 &lsquo;British&rsquo; is annotated with the page [United_Kingdom](http://en.wikipedia.org/wiki/United_Kingdom)

## Problem 1: Evaluation measures

To warm up, we ask you to write code to print the three measures that you will be using for evaluation.

<div class="alert alert-info">

**Write a function that returns the precision, recall, and F1 values** for two given sets of gold-standard entities and predicted entities.

Note that for implementing this function, it doesn’t matter what exactly `gold` and `pred` will contain, except that they will be Python `set` objects.

</div>

In [4]:
def evaluation_scores(gold, pred):
    """Compute precision, recall, and F1 score.
    
    Arguments:
        gold: The set with the gold-standard values.
        pred: The set with the predicted values.
    
    Returns:
        A tuple or list containing the precision, recall, and F1 values
        (in that order), computed based on the specified sets.
    """
    # Count the correctly predicted values:
    correct = len(gold & pred)
    precision = correct / len(pred) if pred else 0.0
    recall = correct / len(gold) if gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return precision, recall, f1

Let's also define a convenience function that displays the scores nicely:

In [5]:
def print_evaluation_scores(result):
    from IPython.display import HTML
    p, r, f = result
    df = pd.DataFrame([{'precision': p, 'recall': r, 'f1-score': f}])
    display(HTML(df.to_html(index=False)))

### 🤞 Test your code

To test your code, you can run the following cell. This should give you a precision of 50%, a recall of 33.3%, and an F1-value of 0.4.

In [6]:
# Some example NER spans, to illustrate how the evaluation function will be used later
example_gold = {("0000-000", 0, 1), ("0000-000", 2, 3), ("0000-000", 6, 7)}
example_pred = {("0000-000", 2, 3), ("0000-000", 6, 8)}

# Compute and print the scores
result = evaluation_scores(example_gold, example_pred)
print_evaluation_scores(result)

# Check if the scores appear correct
with check_solutions() as check:
    check.isclose(result[0], .5)
    check.isclose(result[1], 1. / 3)
    check.isclose(result[2], .4)

precision,recall,f1-score
0.5,0.333333,0.4


## Problem 2: Named entity recognition

One of the first tasks that an information extraction system has to solve is to locate and classify (mentions of) named entities, such as persons and organizations, a task usually known as _named entity recognition (NER)._  For this lab, we will consider a slightly simplified version of NER, by only looking at the _spans_ of tokens containing an entity mention, without the actual entity label.

The English language models in spaCy feature a full-fledged [named entity recognizer](https://spacy.io/usage/linguistic-features#named-entities) that identifies a variety of entities, and can be updated with new entity types by the user.  We therefore start by loading spaCy. _However,_ the data that we will be using has already been tokenized (following the conventions of the [Penn Treebank](https://www.nltk.org/api/nltk.tokenize.treebank.html)), so we need to prevent spaCy from using its own tokenizer on top of this.  We therefore override spaCy&rsquo;s tokenizer with the default one that simply splits on whitespace:

In [7]:
import spacy
from spacy.tokenizer import Tokenizer

nlp = spacy.load('en_core_web_md')    # Let’s use the "medium" (md) model
nlp.tokenizer = Tokenizer(nlp.vocab)  # ...but override the tokenizer

<div class="alert alert-info">

Your task in this problem is to **evaluate the performance of spaCy’s NER component** when predicting entity spans in the _development data_.

This can be done in the following three steps:

1. **Write a function `gold_spans()`** that takes a DataFrame and returns a set of triples of the form `(sentence_id, start_position, end_position)`, one for each entity mention _in the dataset_.
2. **Write a function `pred_spans()`** that takes a DataFrame, runs spaCy’s NER on each sentence, and returns a set of triples (in the same form as above), one for each entity mention _predicted by spaCy_.
3. Evaluate the results using your function from Problem&nbsp;1. _(This code is already given below.)_

</div>

_Note:_ We ask you to implement `gold_spans()` and `pred_spans()` as _generator functions_ that “yield” a single triple at a time, and provide stubs of such functions below that you can use as a starting point. (If you're not familiar with the `yield` keyword in Python, check out [this brief explanation](https://www.nbshare.io/notebook/851988260/Python-Yield/).)

In [8]:
def gold_spans(df):
    """Yield the gold-standard mention spans in a data frame.

    Arguments:
        df: A data frame.

    Yields:
        The gold-standard mention spans in the specified data frame as
        triples consisting of the sentence id, start position, and end
        position of each span.
    """
    # Hint: The Pandas method .itertuples() is useful for iterating over rows in a DataFrame
    # Yield the annotated span for each row:
    for row in df.itertuples():
        yield row.sentence_id, row.beg, row.end

In [9]:
def pred_spans(df):
    """Run and evaluate spaCy's NER.

    Arguments:
        df: A data frame.

    Yields:
        The predicted mention spans in the specified data frame as
        triples consisting of the sentence id, start position, and end
        position of each span.
    """
    # Run NER once for each sentence:
    sentences = df[["sentence_id", "sentence"]].drop_duplicates(subset="sentence_id")
    for row in sentences.itertuples():
        doc = nlp(row.sentence)
        for entity in doc.ents:
            yield row.sentence_id, entity.start, entity.end

#### 🤞 Putting it all together

The following cell shows how you can put it all together and produce the evaluation report, provided you have implemented the functions as generator functions.  You should get a precision above 50%, with a recall above 70%, and an F1-score above 60%.

In [10]:
# Produce the spans
spans_dev_pred = set(pred_spans(df_dev))
spans_dev_gold = set(gold_spans(df_dev))

# Compute and print the evaluation scores
scores = evaluation_scores(spans_dev_gold, spans_dev_pred)
print_evaluation_scores(scores)

precision,recall,f1-score
0.550606,0.775055,0.64383


In [11]:
# Check if the scores appear correct
with check_solutions() as check:
    check(scores[0] > .50, f"Precision should be above 50%, is {scores[0] * 100:.2f}%")
    check(scores[1] > .70, f"Recall should be above 70%, is {scores[1] * 100:.2f}%")
    check.equals(len(spans_dev_gold), 5917)  # The number of unique returned triples is not correct.
    check(('0946-000', 2, 3) in spans_dev_gold, "The first expected triple is not included in the results.")
    check(('1161-010', 1, 3) in spans_dev_gold, "The last expected triple is not included in the results.")

## Problem 3: Error analysis

As you can see in Problem&nbsp;2, the span accuracy of the named entity recognizer is far from perfect. In particular, only slightly more than half of the predicted spans are correct according to the gold standard. Your next task is to analyse this result in more detail.

Let's take a sample of the training data to inspect this way. First, we re-use the functions from Problem&nbsp;2 to obtain the spans:

In [12]:
df_inspect = df_train[:1000]
spans_inspect_gold = set(gold_spans(df_inspect))
spans_inspect_pred = set(pred_spans(df_inspect))

The course module provides a function that builds an interactive viewer for sentences containing _at least one mistake_ (i.e., a false positive, a false negative, or both). It uses spaCy’s span visualizer to highlight each span as one of:

* **TP** (true positive, in green): a predicted span that is also in the gold standard
* **FP** (false positive, in red): a predicted span that is _not_ in the gold standard
* **FN** (false negative, in yellow): a gold-standard span that was _not_ predicted by spaCy

The following cell creates a browser for the sentences containing mistakes that the automated prediction makes.  Use the buttons to step through the sentences one at a time, or enter a number to jump to a specific sentence.  **Try to find any patterns in the mistakes!**  You will use the insights from this manual analysis to implement a post-processing function, as well as for your individual reflection.

In [13]:
from liu_tm.ner import interactive_span_browser

interactive_span_browser(df_inspect, spans_inspect_gold, spans_inspect_pred)

In [14]:
from collections import Counter

# SpaCy labels for predicted spans that are FPs
fp_labels = Counter()
sentences = df_inspect[["sentence_id", "sentence"]].drop_duplicates(subset="sentence_id")
for row in sentences.itertuples():
    doc = nlp(row.sentence)
    for entity in doc.ents:
        span = (row.sentence_id, entity.start, entity.end)
        if span not in spans_inspect_gold:
            fp_labels[entity.label_] += 1

pd.Series(fp_labels).sort_values(ascending=False).rename("false positives")

DATE        157
CARDINAL    149
ORG          48
ORDINAL      27
PERSON       24
NORP         19
GPE          18
MONEY        17
PERCENT      12
QUANTITY      9
EVENT         8
TIME          5
FAC           4
PRODUCT       2
LOC           1
LAW           1
Name: false positives, dtype: int64

In [15]:
fp_labels.keys()

dict_keys(['ORG', 'DATE', 'CARDINAL', 'QUANTITY', 'PERCENT', 'MONEY', 'LOC', 'TIME', 'ORDINAL', 'PERSON', 'GPE', 'PRODUCT', 'EVENT', 'NORP', 'FAC', 'LAW'])

In [ ]:
# Inspect remaining errors after post-processing.
# Each row with overlapping spans is usually a boundary error.
sentences = df_inspect.drop_duplicates(subset="sentence_id").set_index("sentence_id")["sentence"]

def text_of(span):
    sentence_id, start, end = span
    return " ".join(sentences[sentence_id].split()[start:end])

remaining_fp = spans_inspect_pred_improved - spans_inspect_gold
remaining_fn = spans_inspect_gold - spans_inspect_pred_improved
boundary_errors = []
for fp in remaining_fp:
    for fn in remaining_fn:
        same_sentence = fp[0] == fn[0]
        overlap = fp[1] < fn[2] and fn[1] < fp[2]
        if same_sentence and overlap:
            boundary_errors.append({
                "sentence_id": fp[0],
                "spaCy prediction (FP)": text_of(fp),
                "gold span (FN)": text_of(fn),
                "sentence": sentences[fp[0]],
            })

pd.DataFrame(boundary_errors).drop_duplicates()

### Writing a post-processor

Based on your insights from the error analysis above, you should be able to improve the automated prediction that you implemented in Problem&nbsp;2.  In principle, the best way to do this would be to update spaCy&rsquo;s NER model using domain-specific training data.  However, based on an inspection of common errors above, you should be able to find a simpler way to already boost the performance substantially.

<div class="alert alert-info">

**Write code to post-process the output** produced by spaCy so that the predictions more closely align with what’s in our training data.

_Note:_ To filter out specific labels it is useful to know the named entity label scheme, which can be found in the [model's documentation](https://spacy.io/models/en#en_core_web_md).

</div>

In [16]:
def pred_spans_improved(df):
    """Run and evaluate spaCy's NER, with post-processing to improve the results.

    Arguments:
        df: A data frame.

    Yields:
        The predicted mention spans in the specified data frame as
        triples consisting of the sentence id, start position, and end
        position of each span.
    """
    # Ignore numerical and temporal entities that are not annotated in the dataset:
    excluded_labels = {"DATE", "TIME", "PERCENT", "MONEY", "QUANTITY", "ORDINAL", "CARDINAL"}

    sentences = df[["sentence_id", "sentence"]].drop_duplicates(subset="sentence_id")
    for row in sentences.itertuples():
        doc = nlp(row.sentence)
        for entity in doc.ents:
            if entity.label_ not in excluded_labels:
                yield row.sentence_id, entity.start, entity.end

#### 🤞 Test your code

You should be able to obtain an **F1 score of at least 0.80** this way, i.e., a substantial improvement over the scores you got in Problem&nbsp;2.

The following cells report the evaluation measures and test if you achieve the performance goal:

In [17]:
scores_improved = evaluation_scores(spans_dev_gold, set(pred_spans_improved(df_dev)))
print_evaluation_scores(scores_improved)

with check_solutions() as check:
    check(scores_improved[-1] > .8, f"F1-score should be above 0.8 now; got: {scores_improved[-1]:.2f}")

precision,recall,f1-score
0.86166,0.773703,0.815316


In [18]:
spans_inspect_pred_improved = set(pred_spans_improved(df_inspect))

interactive_span_browser(df_inspect,spans_inspect_gold,spans_inspect_pred_improved)

### Storing the outputs of the post-processing

<div class="alert alert-info">

Before moving on, we ask you to **store the outputs of the improved named entity recognizer in a new data frame**. This new frame should have the same layout as the original data frame for the _development data_ that you loaded above, but should contain the *predicted* start and end positions for each token span, rather than the gold positions. As the `label` of each span, you can use the special value `--NME--` for now.

</div>

In [16]:
def df_with_pred_spans(df):
    """Make a new DataFrame with *predicted* NER spans.

    Arguments:
        df: A data frame.

    Returns:
        A *new* data frame with the same layout as `df`, but containing
        the predicted start and end positions for each token span.
    """
    # Look up the sentence text for each predicted span:
    sentences = df.drop_duplicates(subset="sentence_id").set_index("sentence_id")["sentence"]
    rows = []
    for sentence_id, beg, end in pred_spans_improved(df):
        rows.append({"sentence_id": sentence_id, "sentence": sentences[sentence_id],
                     "beg": beg, "end": end, "label": "--NME--"})

    return pd.DataFrame(rows, columns=df.columns)

#### 🤞 Test your code

Run the following cell to run your function and display the first few lines of the new data frame:

In [17]:
df_dev_pred = df_with_pred_spans(df_dev)
display(df_dev_pred.head(10))

with check_solutions() as check:
    check(isinstance(df_dev_pred, pd.DataFrame), "The returned object should be a Pandas DataFrame.")
    # The returned data frame should have the same columns as the original data frame:
    check.equals(df_dev_pred.columns.tolist(), ['sentence_id', 'sentence', 'beg', 'end', 'label'])
    # The returned data frame should only have the special '--NME--' value as label:
    check.equals(df_dev_pred["label"].unique().tolist(), ["--NME--"])

,sentence_id,sentence,beg,end,label
0,0946-001,LONDON 1996-08-30,0,1,--NME--
1,0946-002,West Indian all-rounder Phil Simmons took four...,0,2,--NME--
2,0946-002,West Indian all-rounder Phil Simmons took four...,3,5,--NME--
3,0946-002,West Indian all-rounder Phil Simmons took four...,12,13,--NME--
4,0946-002,West Indian all-rounder Phil Simmons took four...,14,15,--NME--
5,0946-003,"Their stay on top , though , may be short-live...",13,14,--NME--
6,0946-003,"Their stay on top , though , may be short-live...",15,16,--NME--
7,0946-003,"Their stay on top , though , may be short-live...",17,18,--NME--
8,0946-003,"Their stay on top , though , may be short-live...",24,25,--NME--
9,0946-003,"Their stay on top , though , may be short-live...",35,36,--NME--


## Problem 4: Entity linking

Now that we have a method for predicting mention spans, we turn to the task of **entity linking**, which amounts to predicting the knowledge base entity that is referenced by a given mention. In our case, for each span, we want to predict the Wikipedia page that this mention references.

<div class="alert alert-info">

Start by **extending the generator function** that you implemented in Problem&nbsp;2 to yield **labelled spans**.

</div>

In [18]:
def gold_mentions(df):
    """Yield the gold-standard mentions in a data frame.

    Args:
        df: A data frame.

    Yields:
        The gold-standard mention spans in the specified data frame as
        quadruples consisting of the sentence id, start position, end
        position and entity label of each span.
    """
    # Yield the annotated span and label for each row:
    for row in df.itertuples():
        yield row.sentence_id, row.beg, row.end, row.label

#### 🤞 Test your code

To test your code, you can run the following cell, which checks if one of the expected tuples is included in the results:

In [19]:
dev_gold_mentions = set(gold_mentions(df_dev))

with check_solutions() as check:
    check(('1094-020', 0, 1, 'Seattle_Mariners') in dev_gold_mentions, "An expected tuple is not included in the results")
    check.equals(len(dev_gold_mentions), len(spans_dev_gold))  # number of spans should be the same as before

### A baseline for entity linking

A naive baseline for entity linking on our data set is to link each mention span to the Wikipedia page name that we get when we join the tokens in the span by underscores, as is standard in Wikipedia page names. Suppose, for example, that a span contains the two tokens

    Jimi Hendrix

The baseline Wikipedia page name for this span would be

    Jimi_Hendrix

<div class="alert alert-info">

**Implement this naive baseline and evaluate its performance!**

</div>

<div class="alert alert-warning">

**_Important:_** Here and in the remainder of this lab, you should base your experiments on the **predicted spans** that you computed in Problem&nbsp;3!

</div>

In [20]:
def baseline(df):
    """A naive baseline for entity linking that "predicts" Wikipedia
       page names from the tokens in the mention span.

    Arguments:
        df: A data frame.

    Yields:
        The predicted mention spans in the specified data frame as
        quadruples consisting of the sentence id, start position, end
        position and the predicted entity label of each span.
    """
    # Join the tokens in each mention with underscores:
    for row in df.itertuples():
        tokens = row.sentence.split()
        label = "_".join(tokens[row.beg:row.end])
        yield row.sentence_id, row.beg, row.end, label

#### 🤞 Test your code

Again, we can turn to the evaluation measures that we implemented in Problem&nbsp;1.  The expected precision should be around 29%, with an F1-score around 28%.

In [21]:
# Compute and print the evaluation scores
scores = evaluation_scores(dev_gold_mentions, set(baseline(df_dev_pred)))
print_evaluation_scores(scores)

# Check if scores are as expected
with check_solutions() as check:
    check(scores[0] > .28, f"Precision should be above 28%; got {scores[0] * 100:.2f}%")
    check(scores[-1] > .27, f"F1-score should be above 27%; got {scores[-1] * 100:.2f}%")

precision,recall,f1-score
0.296254,0.266013,0.280321


## Problem 5: Better entity linking with a knowledge base

State-of-the-art approaches to entity linking exploit information in _knowledge bases_. In our case, where Wikipedia is the knowledge base, one particularly useful type of information are links to other Wikipedia pages. In particular, we can interpret the anchor texts (the highlighted texts that you click on) as mentions of the entities (pages) that they link to. This allows us to harvest long lists of mention–entity pairings.

The following cell loads a data frame summarizing anchor texts and page references harvested from the first paragraphs of the English Wikipedia. The data frame also contains all entity mentions in the training data (but not the development or the test data).

In [22]:
with bz2.open('kb.tsv.bz2', 'rt') as source:
    df_kb = pd.read_csv(source, sep='\t', quoting=csv.QUOTE_NONE)

To understand what information is available in this data, the following cell shows the entry for the anchor text `Sweden`.

In [23]:
df_kb.loc[df_kb.mention == 'Sweden']

,mention,entity,prob
17436,Sweden,Sweden,0.985768
17437,Sweden,Sweden_national_football_team,0.014173
17438,Sweden,Sweden_men's_national_ice_hockey_team,0.000059


As you can see, each row of the data frame contains a pair $(m, e)$ of a mention $m$ and an entity $e$, as well as the conditional probability $P(e|m)$ for mention $m$ referring to entity $e$. These probabilities were estimated based on the frequencies of mention–entity pairs in the knowledge base. The example shows that the anchor text &lsquo;Sweden&rsquo; is most often used to refer to the entity [Sweden](http://en.wikipedia.org/wiki/Sweden), but in a few cases also to refer to Sweden&rsquo;s national football and ice hockey teams. Note that references are sorted in decreasing order of probability, so that the most probable pairing come first.

<div class="alert alert-info">

**Implement an entity linking method** that resolves each mention to the most probable entity in the data frame. If the mention is not included in the data frame, you can predict the generic label `--NME--` as before.

</div>

In [24]:
def most_probable_method(df, df_kb):
    """An entity linker that resolves each mention to the most probable entity in a knowledge base.

    Arguments:
        df: A data frame containing the mention spans.
        df_kb: A data frame containing the knowledge base.

    Yields:
        The predicted mention spans in the specified data frame as
        quadruples consisting of the sentence id, start position, end
        position and the predicted entity label of each span.
    """
    # Keep the most probable entity for each mention:
    best_entities = df_kb.sort_values("prob", ascending=False).drop_duplicates(subset="mention")
    entity_lookup = dict(zip(best_entities["mention"], best_entities["entity"]))

    for row in df.itertuples():
        mention = " ".join(row.sentence.split()[row.beg:row.end])
        label = entity_lookup.get(mention, "--NME--")
        yield row.sentence_id, row.beg, row.end, label

### 🤞 Test your code

We run the same evaluation as before. The expected precision should now be around 65%, with an F1-score just around 59%.

In [25]:
scores = evaluation_scores(dev_gold_mentions, set(most_probable_method(df_dev_pred, df_kb)))
print_evaluation_scores(scores)

with check_solutions() as check:
    check(scores[0] > .64, f"Precision should be above 64%; got {scores[0] * 100:.2f}%")
    check(scores[-1] > .58, f"F1-score should be above 58%; got {scores[-1] * 100:.2f}%")

precision,recall,f1-score
0.658385,0.591178,0.622974


## Problem 6: Context-sensitive disambiguation

Consider the entity mention &lsquo;Lincoln&rsquo;:

In [26]:
df_kb.loc[df_kb.mention == 'Lincoln']

,mention,entity,prob
10559,Lincoln,"Lincoln,_Nebraska",0.748718
10560,Lincoln,Abraham_Lincoln,0.220513
10561,Lincoln,Lincoln_City_F.C.,0.030769


The most probable entity for this mention turns out to be [Lincoln, Nebraska](http://en.wikipedia.org/wiki/Lincoln,_Nebraska); but in pages about American history, we would be better off to predict [Abraham Lincoln](http://en.wikipedia.org/wiki/Abraham_Lincoln). This suggests that we should try to _disambiguate_ between different entity references based on the textual context on the page from which the mention was taken. Your task in this last problem is to implement this idea.

Set up a dictionary that contains, for each mention $m$ that can refer to more than one entity $e$, a separate Naive Bayes classifier that is trained to predict the correct entity $e$, given the textual context of the mention. As the prior probabilities of the classifier, choose the probabilities $P(e|m)$ that you used in Problem&nbsp;5. To let you estimate the context-specific probabilities, we have compiled a data set with mention contexts:

In [27]:
with bz2.open('contexts.tsv.bz2') as source:
    df_contexts = pd.read_csv(source, sep='\t', quoting=csv.QUOTE_NONE)

This data frame contains, for each ambiguous mention $m$ and each knowledge base entity $e$ to which this mention can refer, up to 100 randomly selected contexts in which $m$ is used to refer to $e$. For this data, a **context** is defined as the 5 tokens to the left and the 5 tokens to the right of the mention. Here are a few examples:

In [28]:
df_contexts.head()

,mention,entity,context
0,1970,UEFA_Champions_League,Cup twice the first in @ and the second in 1983
1,1970,FIFA_World_Cup,America 1975 and during the @ and 1978 World C...
2,1990 World Cup,1990_FIFA_World_Cup,Manolo represented Spain at the @
3,1990 World Cup,1990_FIFA_World_Cup,Hašek represented Czechoslovakia at the @ and ...
4,1990 World Cup,1990_FIFA_World_Cup,renovations in 1989 for the @ The present capa...


Note that, in each context, the position of the mention is indicated by the `@` symbol.

From this data frame, it is easy to select the data that you need to train the classifiers – the contexts and corresponding entities for all mentions. To illustrate this, the following cell shows how to select all contexts that belong to the mention &lsquo;Lincoln&rsquo;:

In [29]:
df_contexts["context"][df_contexts.mention == 'Lincoln']

41465    Nebraska Concealed Handgun Permit In @ municip...
41466    Lazlo restaurants are located in @ and Omaha C...
41467    California Washington Overland Park Kansas @ N...
41468    City Missouri Omaha Nebraska and @ Nebraska It...
41469    by Sandhills Publishing Company in @ Nebraska USA
                               ...                        
41609                                      @ Leyton Orient
41610                    English division three Swansea @ 
41611    league membership narrowly edging out @ on goa...
41612                                          @ Cambridge
41613                                                   @ 
Name: context, Length: 149, dtype: str

<div class="alert alert-info">

**Implement the context-sensitive disambiguation method and evaluate its performance.**

Do this in two parts:
1. Implement a function that builds the classifiers _(refer to the text above for a detailed description)_.
2. Implement a prediction function that uses these classifiers to perform the entity prediction.

</div>

Here are some more _hints_ that may help you along the way:

1. The prior probabilities for a Naive Bayes classifier can be specified using the `class_prior` option. You will have to provide the probabilities in the same order as the alphabetically sorted class (entity) names.

2. Not all mentions in the knowledge base are ambiguous, and therefore not all mentions have context data. If a mention has only one possible entity, pick that one. If a mention has no entity at all, predict the `--NME--` label.

In [30]:
def build_entity_classifiers(df_kb, df_contexts):
    """Build Naive Bayes classifiers for entity prediction.

    Arguments:
        df_kb: A data frame with the knowledge base.
        df_contexts: A data frame with contexts for each mention.

    Returns:
        A dictionary where the keys are mentions and the values are Naive Bayes
        classifiers trained to predict the correct entity, given the textual
        context of the mention (as described in detail above).
    """
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.pipeline import make_pipeline

    classifiers = {}
    for mention, contexts in df_contexts.groupby("mention"):
        entities = sorted(contexts["entity"].unique())
        if len(entities) < 2:
            continue

        # Match the prior probabilities to the sorted entity names:
        probabilities = df_kb[df_kb["mention"] == mention].set_index("entity")["prob"]
        priors = [probabilities[entity] for entity in entities]
        priors = np.array(priors) / sum(priors)
        classifier = make_pipeline(CountVectorizer(), MultinomialNB(class_prior=priors))
        classifier.fit(contexts["context"].fillna(""), contexts["entity"])
        classifiers[mention] = classifier

    return classifiers

In [31]:
def extended_dictionary_method(df, classifiers, df_kb):
    """An entity linker that disambiguates each mention using context-sensitive classifiers,
       falling back to the most probable entity in a knowledge base.

    Arguments:
        df: A data frame containing the mention spans.
        classifiers: A dictionary of classifiers as produced by the
            `build_entity_classifiers` function.
        df_kb: A data frame with the knowledge base. (Should be used
            to look up a mention if it doesn't have a classifier.)

    Yields:
        The predicted mention spans in the specified data frame as
        quadruples consisting of the sentence id, start position, end
        position and the predicted entity label of each span.
    """
    # Use the most probable entity when there is no classifier:
    best_entities = df_kb.sort_values("prob", ascending=False).drop_duplicates(subset="mention")
    entity_lookup = dict(zip(best_entities["mention"], best_entities["entity"]))

    for row in df.itertuples():
        tokens = row.sentence.split()
        mention = " ".join(tokens[row.beg:row.end])
        if mention in classifiers:
            context = tokens[max(0, row.beg - 5):row.beg] + ["@"] + tokens[row.end:row.end + 5]
            label = classifiers[mention].predict([" ".join(context)])[0]
        else:
            label = entity_lookup.get(mention, "--NME--")
        yield row.sentence_id, row.beg, row.end, label

### 🤞 Test your code

The cell below shows how your functions should all come together.

In [32]:
classifiers = build_entity_classifiers(df_kb, df_contexts)
dev_pred_dict_mentions = set(extended_dictionary_method(df_dev_pred, classifiers, df_kb))

with check_solutions() as check:
    check(isinstance(classifiers, dict), "classifiers should be a dictionary")
    check("Lincoln" in classifiers, "The term 'Lincoln' should be in the classifier dictionary")
    check(hasattr(classifiers["Lincoln"], "predict"), "The value of 'Lincoln' should be a classifier or a pipeline (something that has a 'predict' method)")

Finally, the cell below evaluates the results as before. You should expect to see a small (around 1&nbsp;unit) increase in each of precision, recall, and F1.

In [33]:
scores = evaluation_scores(dev_gold_mentions, dev_pred_dict_mentions)
print_evaluation_scores(scores)

with check_solutions() as check:
    check(scores[0] > .65, f"Precision should be above 65%; got {scores[0] * 100:.2f}%")
    check(scores[-1] > .59, f"F1-score should be above 59%; got {scores[-1] * 100:.2f}%")

precision,recall,f1-score
0.674572,0.605712,0.63829


## Individual reflection

<div class="alert alert-warning">

**After you have solved the lab,** save the notebook and **work on the following part individually.**

</div>

<div class="alert alert-info">

**Write a _brief_ reflection on the questions below.**

- You may discuss the questions with your lab partner, but remember to write them up individually.  **Do not copy** answers from your lab partner.
- You may add more cells below (even code cells!) if you wish.

</div>

1. In Problem 3, you performed an error analysis and implemented some post-processing to improve the model’s evaluation scores.  What common errors did you identify, and how did you fix them?  How could you improve the model’s performance further, and what would you need for that?  Discuss this based on **two or three concrete examples** from the error analysis.
2. In Problem 6, how does the “context” data help to disambiguate between different entities?  Can you think of other types of “context” that you could use for disambiguation?  Illustrate this with a **specific example**.

# A1:
Before having implemented the post-processing rule, I checked the distribution of labels that were marked as FPs. I noticed labels like dates, ordinals, percentages that do not really make sense for our usecase as they would not realistically link to any wikipedia articles. So in my post-processing rule I mainly filter out those labels. Those are often recognized by spaCy but those are not the kinds of entities the dataset cares about.
With that the Precision shoots up to about $0.86$, but recall is still somewhat low, so I had to check for FNs, i.e cases that model misses. What I found:

1. spaCy sometimes includes "the" in an entity span wheras the gold standard does not, which causes an exact-span mismatch.

```
[...] a Palestinian helicopter to **the West Bank** [...]
```

This causes 2 types of errors: A FN, because the gold standard expects the span "West Bank" but our model delivers "the West Bank" AND a FP, because "the West Bank" is not considered an entity. That means that our precision and recall are somewhat pessimistic. If we wanted to improve the capabilities, the postprocessor could check for entities that start with "the" , and move the span such that "the" is no longer included.

2. Some all-caps place names and abbreviations are considerd entities by the gold standard but are missed by spaCy, which again reduces recall.

```
LATTAKIA(FN), Aug 10 - waiting time at Lattakia(TP) and [...]
```

```
Afghan(TP) UAE(FN) embassy say [...]
```

Maybe lowercase every text before classifying? That might improve things here.




# A2
Context data can generally help, as the same mention can refer to different intities as the context (in our lab: the surrounding words) gives clues around the intended meaning.

#### As an example: **Jordan**
- **Jordan**: The country
- **Michael Jordan**: The famous american basketball player.
- **Jordan River**

The sourrounding context can often reveal the entity, like basketball, sports point toward the entity **Jordan** being a name, rather than a country.
Context does not need to be the surrounding words. If available, the title /category of the wikipedia article, or other named entities like dates, locations could help infer the entity-type of **Jordan**.

**Congratulations on finishing this lab! 👍**

<div class="alert alert-info">

➡️ Remember that each student should submit their own version of this notebook, with their own individual answers to the reflection questions.

</div>